# Treino do YOLOv8n custom (Sprint 6)

Gera os pesos `models/yolov8n_surgical.pt` consumidos por `src/video/detector.py`. Roda no Google Colab com GPU T4 gratuita. Pipeline auto-contido: baixa CholecSeg8k do Hugging Face, converte mascaras em bounding boxes, treina o YOLOv8n e exporta os artefatos. Idempotente: se o dataset ja existir na pasta configurada, pula direto para o treino.

Contexto da escolha do dataset e do alvo do detector em [docs/arquitetura/decisoes_tecnicas.md](../docs/arquitetura/decisoes_tecnicas.md) (ADR-012).

## 1. Setup

Esta secao prepara o ambiente em 3 partes:

**1.1 GPU + batch:** detecta automaticamente o GPU disponivel via `nvidia-smi` e seleciona o batch/workers apropriado. Suporta T4 (gratuita, 15 GB VRAM) e L4 (Colab Pro, 22-24 GB VRAM, ~2.5x mais rapido). Pode-se forcar manualmente pelo override no codigo.

| GPU | VRAM | Batch padrao | Workers | Velocidade relativa |
|-----|------|--------------|---------|---------------------|
| T4 | 15 GB | 96 | 8 | 1x (baseline) |
| L4 | 22-24 GB | 160 | 12 | ~2.5-3x |

Para mudar o tipo de GPU alocada pelo Colab: `Runtime > Change runtime type > GPU type`. A celula detecta a nova GPU automaticamente.

**1.2 Drive:** todo o estado pesado e cacheado em `MyDrive/medica-ia/`:
- `cholecseg8k_raw_zip/`: zip baixado do HF (3 GB) - evita re-download em sessoes futuras
- `cholecseg8k_yolo/`: dataset convertido pra YOLO (2.7 GB) - evita re-conversao
- `yolo_runs/`: outputs do treino (`best.pt`, `last.pt`, graficos) - evita perder tudo se Colab desconectar

Em re-execucao, o notebook detecta o que ja existe no Drive e pula etapas. O treino tem callback que sincroniza `last.pt` para o Drive ao fim de cada epoch, permitindo retomar de onde parou se a sessao cair.

In [ ]:
# 1.1 GPU + batch (auto-detect)
import subprocess

GPU_CONFIGS = {
    'T4': {'batch': 96, 'workers': 8, 'vram_gb': 15, 'note': 'gratuito'},
    'L4': {'batch': 160, 'workers': 12, 'vram_gb': 22, 'note': 'Colab Pro, ~2.5x mais rapido que T4'},
}

def _detect_gpu():
    """Detecta GPU disponivel via nvidia-smi. Retorna 'T4', 'L4' ou None."""
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            text=True, timeout=10,
        )
        name = out.strip().split('\n')[0].upper()
        for key in ('L4', 'T4'):
            if key in name:
                return key
        print(f'GPU detectada nao mapeada: {name!r}. Usando config T4 (conservadora).')
        return None
    except Exception as exc:
        print(f'Falha ao detectar GPU ({exc}). Usando config T4.')
        return None

# Override manual: descomente pra forcar um tipo (ex: testar limites em outra GPU)
# GPU_TYPE = 'T4'

GPU_TYPE = _detect_gpu() or 'T4'
cfg = GPU_CONFIGS[GPU_TYPE]
BATCH_SIZE = cfg['batch']
WORKERS = cfg['workers']
print(f'GPU detectada: {GPU_TYPE} ({cfg["vram_gb"]} GB VRAM, {cfg["note"]})')
print(f'Batch size: {BATCH_SIZE}, workers: {WORKERS}')

!nvidia-smi
!pip install -q ultralytics==8.3.30 huggingface_hub

# 1.2 Drive (sempre montado, para cache persistente)
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/medica-ia')
DRIVE_RAW_ZIP_DIR = DRIVE_ROOT / 'cholecseg8k_raw_zip'
DRIVE_DATASET = DRIVE_ROOT / 'cholecseg8k_yolo'
DRIVE_RUNS = DRIVE_ROOT / 'yolo_runs'

for d in (DRIVE_ROOT, DRIVE_RAW_ZIP_DIR, DRIVE_DATASET, DRIVE_RUNS):
    d.mkdir(parents=True, exist_ok=True)

# Paths locais no Colab (trabalho mais rapido que no Drive)
DATASET_DIR = '/content/cholecseg8k_yolo'
RAW_DIR = '/content/cholecseg8k_raw'
RUNS_DIR = '/content/runs/detect'
RUN_NAME = 'surgical_instruments'
DATA_YAML = f'{DATASET_DIR}/data.yaml'

print()
print(f'Drive root:           {DRIVE_ROOT}')
print(f'  raw zip cache:      {DRIVE_RAW_ZIP_DIR}')
print(f'  dataset cache:      {DRIVE_DATASET}')
print(f'  runs (output):      {DRIVE_RUNS}')
print(f'Working dir (Colab):  {DATASET_DIR}')


## 2. Dataset

Pipeline em 3 passos: verifica cache + baixa, extrai + converte mascaras em bboxes, gera `data.yaml`. Tudo idempotente.

### 2.1 Cache + download

Pipeline de cache em 2 niveis: primeiro verifica se o dataset convertido ja esta no working dir do Colab; se nao, verifica no Drive (cache persistente). Se em nenhum dos dois, baixa do HF (e salva o zip no Drive para futuras sessoes).

In [ ]:
def split_is_complete(base_dir, split, min_count):
    img_dir = Path(base_dir) / 'images' / split
    if not img_dir.is_dir():
        return False
    return len(list(img_dir.iterdir())) >= min_count

EXPECTED = {'train': 5000, 'val': 1500, 'test': 700}

local_ready = all(split_is_complete(DATASET_DIR, s, n) for s, n in EXPECTED.items())
drive_ready = all(split_is_complete(DRIVE_DATASET, s, n) for s, n in EXPECTED.items())

if local_ready:
    NEEDS_PREPARE = False
    print(f'OK: dataset ja no Colab em {DATASET_DIR}')
elif drive_ready:
    NEEDS_PREPARE = False
    print(f'Dataset encontrado no Drive ({DRIVE_DATASET})')
    print('Copiando para o Colab (working dir mais rapido)...')
    Path(DATASET_DIR).mkdir(parents=True, exist_ok=True)
    !cp -r {DRIVE_DATASET}/. {DATASET_DIR}/
    print('Copia concluida.')
else:
    NEEDS_PREPARE = True
    print(f'Dataset nao encontrado nem no Colab nem no Drive.')

    drive_zip = DRIVE_RAW_ZIP_DIR / 'data' / 'CholecSeg8k.zip'
    if drive_zip.exists():
        print(f'Zip cache no Drive encontrado ({drive_zip.stat().st_size / 1e9:.2f} GB). Reusando.')
        local_zip = Path(RAW_DIR) / 'data' / 'CholecSeg8k.zip'
        local_zip.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(drive_zip, local_zip)
    else:
        print('Baixando CholecSeg8k.zip do HF (3.1 GB)...')
        from huggingface_hub import snapshot_download
        snapshot_download(
            repo_id='minwoosun/CholecSeg8k',
            repo_type='dataset',
            local_dir=RAW_DIR,
            allow_patterns=['data/CholecSeg8k.zip'],
        )
        # Cache no Drive para futuras sessoes
        (DRIVE_RAW_ZIP_DIR / 'data').mkdir(parents=True, exist_ok=True)
        shutil.copy(Path(RAW_DIR) / 'data' / 'CholecSeg8k.zip', drive_zip)
        print(f'Zip cacheado no Drive: {drive_zip}')

    !ls -lh {RAW_DIR}/data/CholecSeg8k.zip


### 2.2 Extracao do zip

Descompacta `CholecSeg8k.zip` (~3 GB) em `RAW_DIR/extracted/`. Usa `zipfile` + `tqdm` para mostrar progresso por arquivo. Tempo esperado no Colab: 3-8 minutos.

In [ ]:
if NEEDS_PREPARE:
    import zipfile
    from pathlib import Path
    from tqdm.auto import tqdm

    zip_path = Path(RAW_DIR) / 'data' / 'CholecSeg8k.zip'
    extracted_root = Path(RAW_DIR) / 'extracted'

    if not extracted_root.exists() or not any(extracted_root.iterdir()):
        extracted_root.mkdir(parents=True, exist_ok=True)
        print(f'Extraindo {zip_path} -> {extracted_root}')
        with zipfile.ZipFile(zip_path) as zf:
            members = zf.namelist()
            for member in tqdm(members, desc='Extraindo', unit='arq'):
                zf.extract(member, extracted_root)
        print(f'Extracao concluida: {len(members)} arquivos em {extracted_root}.')
    else:
        print(f'Zip ja extraido em {extracted_root}, pulando.')
else:
    print('Pulado (cache).')


### 2.3 Conversao mask -> bbox YOLO

Indexa os pares (frame, mascara) extraidos, faz split estratificado 70/20/10 com seed fixa (42), e converte cada mascara grayscale em bounding boxes YOLO. **3 classes alvo:**
- `grasper` (pixel 31) - pinca atraumatica laparoscopica
- `l_hook_electrocautery` (pixel 32) - gancho eletrocauterizador
- `blood` (pixel 24) - sangramento durante o procedimento

Bboxes menores que 100 px sao descartadas. A classe `blood` foi adicionada como sinal de complicacao cirurgica (alinhada com 'Sinais de complicacoes em cirurgias ginecologicas' do enunciado), nao apenas como descricao de instrumental.

In [ ]:
if NEEDS_PREPARE:
    import shutil
    from pathlib import Path
    import cv2
    import numpy as np
    from tqdm.auto import tqdm

    extracted_root = Path(RAW_DIR) / 'extracted'

    INSTRUMENT_CLASSES = {'grasper': 0, 'l_hook_electrocautery': 1, 'blood': 2}
    CLASS_PIXEL_VALUE = {'grasper': 31, 'l_hook_electrocautery': 32, 'blood': 24}
    SPLIT_RATIO = (0.7, 0.2, 0.1)
    SEED = 42
    MIN_BBOX_AREA_PX = 100

    def mask_to_bboxes_by_class(mask_path):
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            return {}
        bboxes = {}
        for class_name, pixel_val in CLASS_PIXEL_VALUE.items():
            ys, xs = np.where(mask == pixel_val)
            if len(xs) == 0:
                continue
            x_min, x_max = int(xs.min()), int(xs.max())
            y_min, y_max = int(ys.min()), int(ys.max())
            w, h = x_max - x_min, y_max - y_min
            if w * h < MIN_BBOX_AREA_PX:
                continue
            bboxes[class_name] = (x_min, y_min, w, h)
        return bboxes

    def yolo_format(bbox, img_w, img_h, class_id):
        x, y, w, h = bbox
        cx = (x + w / 2) / img_w
        cy = (y + h / 2) / img_h
        return f'{class_id} {cx:.6f} {cy:.6f} {w / img_w:.6f} {h / img_h:.6f}\n'

    out = Path(DATASET_DIR)
    for split in ('train', 'val', 'test'):
        (out / 'images' / split).mkdir(parents=True, exist_ok=True)
        (out / 'labels' / split).mkdir(parents=True, exist_ok=True)

    print('Indexando pares (frame, mask)...')
    items = []
    for img_path in tqdm(sorted(extracted_root.rglob('frame_*_endo.png')), desc='Indexando', unit='arq'):
        if any(s in img_path.name for s in ('color_mask', 'watershed_mask')):
            continue
        ann_path = img_path.with_name(img_path.name.replace('_endo.png', '_endo_mask.png'))
        if ann_path.exists():
            items.append((img_path, ann_path))

    if not items:
        raise RuntimeError(
            f'Nenhum par (frame, mask) encontrado em {extracted_root}. '
            f'Conferir layout do zip extraido (esperado: subpastas video_XX/video_XX_YY/ com frame_*_endo.png e frame_*_endo_mask.png).'
        )

    print(f'Pares encontrados: {len(items)}')

    rng = np.random.default_rng(seed=SEED)
    indices = list(range(len(items)))
    rng.shuffle(indices)
    n = len(items)
    n_train = int(n * SPLIT_RATIO[0])
    n_val = int(n * SPLIT_RATIO[1])
    splits = {
        'train': indices[:n_train],
        'val': indices[n_train:n_train + n_val],
        'test': indices[n_train + n_val:],
    }

    cls_count = {cid: 0 for cid in INSTRUMENT_CLASSES.values()}
    for split_name, idxs in splits.items():
        for i in tqdm(idxs, desc=f'Convertendo {split_name}', unit='frame'):
            img_path, ann_path = items[i]
            unique_name = f'{img_path.parent.name}_{img_path.name}'
            unique_stem = f'{img_path.parent.name}_{img_path.stem}'
            shutil.copy(img_path, out / 'images' / split_name / unique_name)
            dst_lbl = out / 'labels' / split_name / f'{unique_stem}.txt'

            img = cv2.imread(str(img_path))
            if img is None:
                dst_lbl.touch()
                continue
            h, w = img.shape[:2]

            bboxes = mask_to_bboxes_by_class(ann_path)
            lines = []
            for class_name, bbox in bboxes.items():
                cid = INSTRUMENT_CLASSES[class_name]
                lines.append(yolo_format(bbox, w, h, cid))
                cls_count[cid] += 1
            dst_lbl.write_text(''.join(lines) if lines else '')

    print(f'\nResumo: {n} frames processados')
    for name, cid in sorted(INSTRUMENT_CLASSES.items(), key=lambda kv: kv[1]):
        print(f'  {name} (id {cid}): {cls_count[cid]} bboxes')
    print(f"Splits: train={len(splits['train'])} val={len(splits['val'])} test={len(splits['test'])}")

    # Sincronizar dataset convertido pro Drive (cache persistente)
    print(f'\nSincronizando dataset convertido com o Drive ({DRIVE_DATASET})...')
    !rm -rf {DRIVE_DATASET}
    !cp -r {DATASET_DIR}/ {DRIVE_DATASET}
    print('Sync concluido. Em proximas sessoes, dataset sera carregado do Drive.')
else:
    print('Pulado (cache).')


### 2.4 data.yaml

Aponta para a raiz do dataset no Colab (sobrescreve o path relativo do repo local).

In [ ]:
import yaml

with open(DATA_YAML, 'w') as f:
    yaml.safe_dump({
        'path': DATASET_DIR,
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'nc': 3,
        'names': {0: 'grasper', 1: 'l_hook_electrocautery', 2: 'blood'},
    }, f, sort_keys=False)

print(open(DATA_YAML).read())


### 2.5 Validacao + visualizacao

Sanity check: contagem por split e quatro amostras com bbox sobre o frame.

In [ ]:
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(DATASET_DIR, 'images', split)
    lbl_dir = os.path.join(DATASET_DIR, 'labels', split)
    n_imgs = len(os.listdir(img_dir))
    n_nonempty = sum(1 for f in os.listdir(lbl_dir) if os.path.getsize(os.path.join(lbl_dir, f)) > 0)
    print(f'{split:>5}: {n_imgs:>5} imagens ({n_nonempty} com bbox)')


In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

random.seed(42)

CLASS_COLORS = {0: 'lime', 1: 'magenta', 2: 'red'}
CLASS_LABELS = {0: 'grasper', 1: 'l_hook', 2: 'blood'}

img_dir = os.path.join(DATASET_DIR, 'images', 'train')
lbl_dir = os.path.join(DATASET_DIR, 'labels', 'train')

candidates = [f for f in os.listdir(img_dir)
              if os.path.getsize(os.path.join(lbl_dir, f.replace('.png', '.txt'))) > 0]

if not candidates:
    raise RuntimeError(
        f'Nenhuma imagem com bbox em {lbl_dir}. '
        'A conversao da 2.2 nao gerou labels (ou o cache em DATASET_DIR ficou inconsistente). '
        'Apague DATASET_DIR e rode tudo de novo a partir da 2.1.'
    )

samples = random.sample(candidates, min(4, len(candidates)))
n = len(samples)
rows = (n + 1) // 2
fig, axes = plt.subplots(rows, 2, figsize=(14, 5 * rows))
axes_list = list(axes.flat) if n > 1 else [axes]

for ax, name in zip(axes_list, samples):
    img = Image.open(os.path.join(img_dir, name))
    W, H = img.size
    ax.imshow(img)
    ax.set_title(name, fontsize=9)
    ax.axis('off')

    with open(os.path.join(lbl_dir, name.replace('.png', '.txt'))) as f:
        for line in f:
            parts = line.strip().split()
            cls = int(parts[0])
            xc, yc, w, h = map(float, parts[1:5])
            x0 = (xc - w / 2) * W
            y0 = (yc - h / 2) * H
            ax.add_patch(patches.Rectangle(
                (x0, y0), w * W, h * H,
                linewidth=2, edgecolor=CLASS_COLORS.get(cls, 'cyan'), facecolor='none',
            ))
            ax.text(x0, max(0, y0 - 5), CLASS_LABELS.get(cls, str(cls)),
                    color=CLASS_COLORS.get(cls, 'cyan'), fontsize=9,
                    bbox=dict(facecolor='black', alpha=0.6, pad=2))

for ax in axes_list[n:]:
    ax.axis('off')

plt.tight_layout()
plt.show()


## 3. Treino

Treino na GPU configurada na celula 1.1 (`device=0` no `model.train()`). Partimos dos pesos `yolov8n.pt` (pre-treinado em COCO) e fazemos fine-tuning nas **3 classes alvo** (grasper, l_hook_electrocautery, blood). Escolhemos a variante **nano** porque o app Gradio do projeto roda os pesos finais em CPU local (sem GPU dedicada), entao a inferencia precisa ser leve. O treino em si nao tem essa restricao: usa a GPU do Colab.

`BATCH_SIZE` e `WORKERS` vem das constantes definidas em 1.1 (96/8 para T4, 160/12 para L4). Os valores foram calibrados para usar ~70-80% da VRAM disponivel sem risco de OOM.

**Resilencia a desconexao do Colab:**
- Callback `on_train_epoch_end` copia `last.pt` e `best.pt` pro Drive ao fim de cada epoch
- Se a sessao cair, ao re-abrir o notebook o codigo detecta o checkpoint no Drive e oferece resumir via `resume=True`
- Sync final completo (graficos, csv, weights) ao terminar o treino

In [ ]:
from ultralytics import YOLO

# Detectar checkpoint anterior no Drive (caso sessao tenha caido)
drive_last = DRIVE_RUNS / RUN_NAME / 'weights' / 'last.pt'
local_run_dir = Path(RUNS_DIR) / RUN_NAME

resume_flag = False
if drive_last.exists() and not (local_run_dir / 'weights' / 'last.pt').exists():
    size_mb = drive_last.stat().st_size / 1e6
    print(f'Checkpoint anterior no Drive ({size_mb:.1f} MB): {drive_last}')
    print('Copiando run dir do Drive para Colab e resumindo...')
    shutil.copytree(DRIVE_RUNS / RUN_NAME, local_run_dir, dirs_exist_ok=True)
    model = YOLO(str(local_run_dir / 'weights' / 'last.pt'))
    resume_flag = True
else:
    model = YOLO('yolov8n.pt')

# Callback: ao fim de cada epoch, copiar weights/last.pt e weights/best.pt pro Drive
def _sync_weights_to_drive(trainer):
    save_dir = Path(trainer.save_dir)
    dst_weights = DRIVE_RUNS / RUN_NAME / 'weights'
    dst_weights.mkdir(parents=True, exist_ok=True)
    for wname in ('last.pt', 'best.pt'):
        src = save_dir / 'weights' / wname
        if src.exists():
            shutil.copy(src, dst_weights / wname)

model.add_callback('on_train_epoch_end', _sync_weights_to_drive)

# Treinar (ou resumir)
if resume_flag:
    results = model.train(resume=True)
else:
    results = model.train(
        data=DATA_YAML,
        epochs=40,
        imgsz=640,
        batch=BATCH_SIZE,
        workers=WORKERS,
        device=0,
        name=RUN_NAME,
        patience=10,
        project=RUNS_DIR,
        exist_ok=True,
        verbose=True,
    )

# Sync final completo (graficos, csv, etc) pro Drive
print('\nSincronizando run dir completo com o Drive...')
dst = DRIVE_RUNS / RUN_NAME
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(local_run_dir, dst)
print(f'OK: {dst}')


**Esperado:** `mAP50 > 0.7` ao fim do treino. Se travar abaixo de 0.3 nas primeiras 5 epocas, revisar splits/labels.

## 4. Avaliacao

Metricas no split de teste + curvas + matriz de confusao + predicoes visuais.

In [ ]:
best_path = f'{RUNS_DIR}/{RUN_NAME}/weights/best.pt'
best_model = YOLO(best_path)

metrics = best_model.val(
    data=DATA_YAML,
    split='test',
    project=RUNS_DIR,
    name=f'{RUN_NAME}_test',
    exist_ok=True,
)

print(f'mAP50:     {metrics.box.map50:.4f}')
print(f'mAP50-95:  {metrics.box.map:.4f}')
print(f'precision: {metrics.box.mp:.4f}')
print(f'recall:    {metrics.box.mr:.4f}')
print()
print('mAP50 por classe:')
for i, name in metrics.names.items():
    print(f'  {name:<25s} {metrics.box.maps[i]:.4f}')


In [ ]:
from IPython.display import Image as IPImage, display

display(IPImage(f'{RUNS_DIR}/{RUN_NAME}/results.png'))
display(IPImage(f'{RUNS_DIR}/{RUN_NAME}_test/confusion_matrix.png'))


In [ ]:
test_img_dir = os.path.join(DATASET_DIR, 'images', 'test')
test_samples = random.sample(os.listdir(test_img_dir), 4)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, name in zip(axes.flat, test_samples):
    img_path = os.path.join(test_img_dir, name)
    pred = best_model(img_path, conf=0.25, verbose=False)[0]
    annotated = pred.plot()
    ax.imshow(annotated[..., ::-1])
    ax.set_title(name, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()


### 4.5 Validacao visual: video anotado com Blood detectado

Cura uma sequencia de frames do **test split** que contenham a classe `Blood` (id=2), roda o `best.pt` em cada frame, anota com bboxes detectadas e empacota em MP4. Esse video serve como evidencia visual no relatorio tecnico (Secao 7 - Validacao tecnica do modelo) de que o YOLO custom efetivamente detecta sangramento em frames de cirurgia laparoscopica.

O MP4 vai pro Drive em `yolo_runs/<RUN_NAME>/validation_blood_detected.mp4`.

In [ ]:
import re
import subprocess
import tempfile
import shutil as _shutil
from collections import defaultdict
from pathlib import Path
import cv2

# 1) Encontrar frames do test split com label de Blood (classe 2)
test_lbl_dir = Path(DATASET_DIR) / 'labels' / 'test'
test_img_dir = Path(DATASET_DIR) / 'images' / 'test'

blood_frames = []
for lbl_file in sorted(test_lbl_dir.iterdir()):
    if not lbl_file.is_file():
        continue
    content = lbl_file.read_text()
    has_blood = any(line.strip().startswith('2 ') for line in content.split('\n'))
    if has_blood:
        blood_frames.append(lbl_file.stem + '.png')

print(f'Frames de test com Blood: {len(blood_frames)}')

if not blood_frames:
    print('AVISO: nenhum frame com classe Blood no test split.')
    print('Provavel causa: dataset convertido sem a classe 2. Refazer a 2.3 com 3 classes.')
else:
    # 2) Agrupar por sequencia de video original (extrair video_XX_NN do prefixo)
    def _seq_id(name):
        m = re.match(r'(video_\d+_\d+)_', name)
        return m.group(1) if m else None

    def _frame_num(name):
        m = re.search(r'frame_(\d+)', name)
        return int(m.group(1)) if m else -1

    by_seq = defaultdict(list)
    for name in blood_frames:
        s = _seq_id(name)
        if s:
            by_seq[s].append(name)

    # 3) Pegar a sequencia com mais frames de Blood
    best_seq = max(by_seq.items(), key=lambda kv: len(kv[1]))
    seq_name, seq_frames = best_seq
    seq_frames = sorted(seq_frames, key=_frame_num)
    print(f'Sequencia escolhida: {seq_name} ({len(seq_frames)} frames com Blood)')

    # 4) Pegar ate 30 frames consecutivos
    sample = seq_frames[:30]
    print(f'Selecionados {len(sample)} frames pra video anotado')

    # 5) Rodar inferencia + anotar
    tmp_dir = Path(tempfile.mkdtemp(prefix='annot_'))
    try:
        for i, fname in enumerate(sample):
            img_path = test_img_dir / fname
            pred = best_model(str(img_path), conf=0.25, verbose=False)[0]
            annotated = pred.plot()  # numpy BGR com bboxes desenhadas
            cv2.imwrite(str(tmp_dir / f'frame_{i:04d}.png'), annotated)

        # 6) Empacotar em MP4 via ffmpeg @ 8 fps (frames sao da mesma sequencia mas nao consecutivos no tempo)
        mp4_out = DRIVE_RUNS / RUN_NAME / 'validation_blood_detected.mp4'
        mp4_out.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            [
                'ffmpeg', '-y',
                '-framerate', '8',
                '-i', str(tmp_dir / 'frame_%04d.png'),
                '-c:v', 'libx264',
                '-pix_fmt', 'yuv420p',
                str(mp4_out),
            ],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        size_mb = mp4_out.stat().st_size / (1024 * 1024)
        print(f'\nVideo anotado salvo: {mp4_out}')
        print(f'Tamanho: {size_mb:.2f} MB ({len(sample)} frames @ 8 fps)')
        print('Use esse MP4 como evidencia visual no relatorio (Secao 7).')
    finally:
        _shutil.rmtree(tmp_dir, ignore_errors=True)


## 5. Export

Copia `best.pt` + graficos para `EXPORT_DIR`. Baixar `best.pt` pela barra lateral do Colab (ou pelo Drive se `USE_DRIVE=True`), colocar em `models/yolov8n_surgical.pt` no repo e ajustar `YOLO_WEIGHTS_PATH` no `.env`.

In [ ]:
# Os artefatos ja foram sincronizados pelo callback do treino.
# Esta celula confirma o que esta no Drive em yolo_runs/<RUN_NAME>/

drive_run = DRIVE_RUNS / RUN_NAME
print(f'Artefatos no Drive ({drive_run}):')
print()

key_files = [
    'weights/best.pt',
    'weights/last.pt',
    'results.png',
    'results.csv',
    f'../{RUN_NAME}_test/confusion_matrix.png',
    'val_batch0_pred.jpg',
]

for rel in key_files:
    f = drive_run / rel
    if f.exists():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f'  OK   {rel:<40s} ({size_mb:.2f} MB)')
    else:
        print(f'  SKIP {rel:<40s} (nao gerado)')


## Fontes

- CholecSeg8k: Hong et al. (2020), [arXiv:2012.12463](https://arxiv.org/abs/2012.12463) (CC BY-NC-SA 4.0)
- Ultralytics YOLOv8: [docs.ultralytics.com](https://docs.ultralytics.com/)
- Repo do projeto: [github.com/joalissonborges94/medica-ia-multimodal](https://github.com/joalissonborges94/medica-ia-multimodal)